In [1]:
# Install the bridge between Wav2Vec2 and KenLM
!pip install pyctcdecode
!pip install https://github.com/kpu/kenlm/archive/master.zip
!pip install librosa transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.0/535.0 kB 5.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 30.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pyctcdecode] [hypothesis]
     | 553.6 kB 3.0 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.2.0-cp39-cp39-linux_x86_64.whl size=606409 sha256=2b609e88bc866bc99d20892df380ff0e6a7d76584eba479f6e122ffca0a8ce5e
  Stored in directory: /tmp/pip-ephem-wheel-cache-g14f7dvw/wheels/b5/52/c9/af2949d9776846ea81a9cba52a4fe5a81b9ace3b9f2530c3f3
Successfully built kenlm


In [1]:
import torch
import librosa
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder


In [28]:
# PATHS of Drive locations
import os
import torch
import librosa
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor

# 1. Define your paths (Adjust folder names as needed)
project_root = Path.home() / "MalteseProject"
MODEL_PATH = project_root / "Runs" / "SplitRunReduced" / "checkpoint-2780"
LM_PATH = project_root / "3_tokenised.bin"


MODEL_PATH = str(MODEL_PATH)
LM_PATH = str(LM_PATH)
ARPA_PATH = str(project_root / "3_tok_pros.arpa")

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def get_unigram_list(arpa_path):
    unigrams = []
    with open(arpa_path, "r") as f:
        is_unigram_section = False
        for line in f:
            if "\\1-grams:" in line:
                is_unigram_section = True
                continue
            if "\\2-grams:" in line or line.startswith("\\end\\"):
                break
            if is_unigram_section:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[1]
                    if word not in ["<s>", "</s>", "<unk>"]:
                        unigrams.append(word)
    return unigrams

print("Extracting unigrams...")
unigrams = get_unigram_list(ARPA_PATH)

Extracting unigrams...


In [29]:
# 1. Load Model & Processor
model = Wav2Vec2BertForCTC.from_pretrained(MODEL_PATH).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL_PATH)

# 2. Build the Decoder (The linguistic brain)
vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

decoder = build_ctcdecoder(
    labels=sorted_vocab,
    kenlm_model_path=LM_PATH,
    unigrams=unigrams,
    alpha=0.7, # Weight of the LM. Try 0.5 to 0.7
    beta=3 # Weight for word insertion
)
print("✅ Model and KenLM Decoder Loaded!")

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


✅ Model and KenLM Decoder Loaded!


In [16]:
def run_comparison(audio_path):
    # Load and Pre-process
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    # Get Logits
    with torch.no_grad():
        logits = model(input_features).logits

    # --- VERSION A: GREEDY (What the model literally hears) ---
    pred_ids = torch.argmax(logits, dim=-1)
    greedy_text = processor.batch_decode(pred_ids)[0]

    # --- VERSION B: KENLM (What the model + Maltese logic says) ---
    logits_numpy = logits.cpu().numpy()[0]
    kenlm_text = decoder.decode(logits_numpy, beam_width=128)

    print(f"\nTarget Audio: {audio_path}")
    print(f"RAW MODEL:   {greedy_text}")
    print(f"WITH KENLM:  {kenlm_text}")
    print("-" * 30)

In [27]:
# The /mnt/c/ tells Linux to look at your Windows C: drive
AUDIO_DIR = Path("/mnt/c/Users/Ian/Documents/TeziTesting/SampleAudios")

# Now this call will look in Windows, pull the audio into WSL, 
# and run it through your Linux-based model.
run_comparison(str(AUDIO_DIR / "Guze.wav"))


Target Audio: /mnt/c/Users/Ian/Documents/TeziTesting/SampleAudios/Guze.wav
RAW MODEL:   ġuże bonnici staqsa lil ġesu pero ma wieġbu ħaddir- responsabbiltà għall- identità tiegħu kien qed jidgħa fuqu
WITH KENLM:  ġuże bonnici staqsa lil ġesù pero ma wieġbu ħadd ir- responsabbiltà għall- identità tiegħu kien qed jidgħa fuqu
------------------------------


In [24]:
# The /mnt/c/ tells Linux to look at your Windows C: drive
AUDIO_DIR = Path("/mnt/c/Users/Ian/Documents/TeziTesting/SampleAudios")

# Now this call will look in Windows, pull the audio into WSL, 
# and run it through your Linux-based model.
run_comparison(str(AUDIO_DIR / "Raptest.wav"))


Target Audio: /mnt/c/Users/Ian/Documents/TeziTesting/SampleAudios/Raptest.wav
RAW MODEL:   ħa jdilkom kif beda kollox tgħaddien il- kedda kont ormfaj żgar fil- claien id- daj għax mhux niffoka kollox imċajpear noħroġ u nidħol ġol- lokki ma niħa ajtgħidlil broodu għax dejjem għandi l lat
WITH KENLM:  ħa idilkom kif beda kollox tħaddin il- kedda kont fil- kain id- dar għax mhux niffoka kollox imċajpar noħroġ u nidħol ġol- loki ma na għidli brodu għax dejjem għandi l- lat
------------------------------
